In [1]:
import zarr
import dask.array as da
import pandas as pd
import numpy as np
import napari
import os
from tqdm.auto import tqdm

In [2]:
df = pd.read_pickle('/mnt/OPERA3/Nathan/data/macrohet/macrohet_results/dfs/sc_df.pkl')

In [3]:
df.keys()

Index(['Time (hours)', 'Mtb Area (µm)', 'dMtb Area (µm)', 'Mphi Area (µm)',
       'dMphi Area (µm)', 'Infection Status', 'Initial Infection Status',
       'Final Infection Status', 'x', 'y', 'GFP', 'RFP', 'Eccentricity', 'MSD',
       'Technical Replicate', 'Biological Replicate', 'Strain', 'Compound',
       'Concentration', 'Cell ID', 'Acquisition ID', 'Experiment ID',
       'Unique ID', 'ID', 'Edge Status', 'Uptake',
       'dMtb Area between frames (µm)', 'Mtb Area Processed (µm)',
       'Time Model (hours)', 'Mtb Area Model (µm)', 'mtb_origin',
       'Doubling Amounts', 'Doubling Times', 'r2', 'Frame', 'category_rank'],
      dtype='object')

In [4]:
subset_df = df[df['mtb_origin']=='Growth']
subset_df

,Time (hours),Mtb Area (µm),dMtb Area (µm),Mphi Area (µm),dMphi Area (µm),Infection Status,Initial Infection Status,Final Infection Status,x,y,...,dMtb Area between frames (µm),Mtb Area Processed (µm),Time Model (hours),Mtb Area Model (µm),mtb_origin,Doubling Amounts,Doubling Times,r2,Frame,category_rank
405,0.0,46.797680,136.772588,660.776979,-68.386294,NaN,1.0,1.0,519.922607,876.779602,...,NaN,NaN,NaN,NaN,Growth,"[50.6, 101.2]",[28.0],0.97,0,NaN
406,1.0,48.719647,136.772588,585.105086,-68.386294,NaN,1.0,1.0,522.290833,876.766357,...,1.921968,NaN,NaN,NaN,Growth,"[50.6, 101.2]",[28.0],0.97,1,NaN
407,2.0,52.206007,136.772588,582.020998,-68.386294,True,1.0,1.0,524.336243,874.563110,...,3.486360,NaN,2.0,47.625714,Growth,"[50.6, 101.2]",[28.0],0.97,2,NaN
408,3.0,50.552221,136.772588,572.232372,-68.386294,True,1.0,1.0,516.952454,876.656799,...,-1.653786,NaN,3.0,49.099089,Growth,"[50.6, 101.2]",[28.0],0.97,3,NaN
409,4.0,54.463202,136.772588,590.669853,-68.386294,True,1.0,1.0,521.947449,880.909363,...,3.910981,50.552221,4.0,50.593160,Growth,"[50.6, 101.2]",[28.0],0.97,4,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1906240,67.0,69.324929,NaN,1343.164920,NaN,NaN,0.0,1.0,1186.000000,1359.000000,...,NaN,6.704539,NaN,NaN,Growth,"[1.92, 3.84]",[5.0],0.85,134,NaN
1906241,67.5,73.146516,NaN,1805.621646,NaN,NaN,0.0,1.0,1158.000000,1359.000000,...,NaN,25.600163,NaN,NaN,Growth,"[1.92, 3.84]",[5.0],0.85,135,NaN
1906242,68.0,NaN,NaN,NaN,NaN,NaN,0.0,1.0,1228.000000,1372.000000,...,NaN,40.366910,NaN,NaN,Growth,"[1.92, 3.84]",[5.0],0.85,136,NaN
1906243,68.5,0.000000,NaN,361.374632,NaN,NaN,0.0,1.0,1235.000000,1368.000000,...,NaN,40.366910,NaN,NaN,Growth,"[1.92, 3.84]",[5.0],0.85,137,NaN


In [5]:
n_tracks = subset_df.groupby(['Experiment ID', 'Acquisition ID']).size()

In [6]:
n_tracks.nlargest(1)

Experiment ID  Acquisition ID
PS0000         (4, 5)            3883
dtype: int64

In [7]:
image_fn = '/mnt/OPERA3/Nathan/data/macrohet/PS0000/acquisition/zarr/(4, 5).zarr'
images = da.from_zarr(f"{image_fn}/images").max(axis=2)

In [8]:
images

dask.array<max-aggregate, shape=(75, 2, 6048, 6048), dtype=uint16, chunksize=(1, 1, 6048, 6048), chunktype=numpy.ndarray>

In [9]:
# Selects rows where 'Experiment ID' equals 'PS0000' AND 'Acquisition ID' equals '(4, 5)'.
tracks = df[(df['Experiment ID'] == 'PS0000') & (df['Acquisition ID'] == (4, 5) )][['Cell ID','Frame','y','x']].dropna().to_numpy(dtype=np.float64)
tracks

array([[1006.        ,    4.        , 1189.13110352,  926.14569092],
       [1006.        ,    5.        , 1176.62390137,  931.15734863],
       [1006.        ,    6.        , 1190.92285156,  946.45581055],
       ...,
       [ 996.        ,   72.        ,  684.67504883,  639.98370361],
       [ 996.        ,   73.        ,  688.83764648,  635.66717529],
       [ 996.        ,   74.        ,  685.81958008,  635.29003906]],
      shape=(16575, 4))

In [10]:
camera_data_tuples = [
    ('center', (39.75, 3023.5, 3023.5)),
    ('zoom', 0.139484126984127),
    ('angles', (30.987817719483903, 35.572853672744216, 48.907463346476064)),
    ('perspective', 0.0),
]

# Convert the list of (key, value) tuples directly into a dictionary.
view_dictionary = dict(camera_data_tuples)
view_dictionary

{'center': (39.75, 3023.5, 3023.5),
 'zoom': 0.139484126984127,
 'angles': (30.987817719483903, 35.572853672744216, 48.907463346476064),
 'perspective': 0.0}

In [11]:
%%time
pad_frame = np.zeros((2, 6048, 6048))

CPU times: user 125 μs, sys: 71 μs, total: 196 μs
Wall time: 210 μs


In [12]:
pad_frame

array([[[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]]], shape=(2, 6048, 6048))

In [13]:
np.stack([images[frame_n], pad_frame])

NameError: name 'frame_n' is not defined

In [14]:

v = napari.Viewer(title = 'actual execution of animation')
print("DEBUG: 1. Viewer initialized.") # Tracepoint 1

for frame_n in tqdm(range(len(images))):
    print(f"\nDEBUG: 2. --- Starting Frame: {frame_n} ---") # Tracepoint 2
    screenshot_path = f'/mnt/OPERA3/Nathan/hpig_tracking_animation/t{frame_n}.png'
    if os.path.exists(screenshot_path):
        continue

    # --- Image Layer Management ---
    if frame_n == 0:
        img_scale = (1,1,1)
        print("DEBUG: 3. Frame 0: img_scale set to (1,1,1).") # Tracepoint 3
    else:
        img_scale=(1,1,1)
        print("DEBUG: 4. Frame > 0: img_scale set to (1,1,1). Starting layer pop.") # Tracepoint 4
        
        # Pop existing layers
        for i in reversed(range(len(v.layers))):
            v.layers.pop(i) 
            print(f"DEBUG: 5. Layer at index {i} popped.") # Tracepoint 5
            
    try:
        napari_images = np.stack([images[frame_n], pad_frame])
        v.add_image(napari_images, channel_axis=1, colormap=['green','magenta'], )#scale=img_scale)
        print(f"DEBUG: 6. Image layer added up to frame {frame_n}.") # Tracepoint 6
    except Exception as e:
        print(f"ERROR: Image layer failed to add at frame {frame_n}: {e}")
        continue
        
    if frame_n==0:
        print("DEBUG: 7. Frame 0: Skipping track and camera update.") # Tracepoint 7
        continue
        
    # --- Tracks Filtering and Conversion ---
    try:
        current_tracks = df[(df['Experiment ID'] == 'PS0000') 
                          & (df['Acquisition ID'] == (4, 5) ) 
                          & (df['Frame'] <= frame_n )][['Cell ID','Frame','y','x']].dropna().to_numpy(dtype=np.float64)
        print(f"DEBUG: 8. Tracks filtered and converted to numpy. Shape: {current_tracks.shape}") # Tracepoint 8
    except Exception as e:
        print(f"ERROR: Tracks data preparation failed at frame {frame_n}: {e}")
        continue
        
    try:
        v.add_tracks(current_tracks, scale = (80, 5.04, 5.04))
        print("DEBUG: 9. Tracks layer added.") # Tracepoint 9
    except Exception as e:
        print(f"ERROR: Tracks layer failed to add at frame {frame_n}: {e}")
        
    # --- Camera and View Setup ---
    v.dims.ndisplay = 3
    print("DEBUG: 10. NDisplay set to 3.") # Tracepoint 10
    
    # Apply camera settings from the dictionary
    v.camera.center = view_dictionary['center']
    print(f"DEBUG: 11. Camera Center set to: {view_dictionary['center']}") # Tracepoint 11
    v.camera.zoom = view_dictionary['zoom']
    print(f"DEBUG: 12. Camera Zoom set to: {view_dictionary['zoom']}") # Tracepoint 12
    v.camera.angles = view_dictionary['angles']
    print(f"DEBUG: 13. Camera Angles set to: {view_dictionary['angles']}") # Tracepoint 13
    v.camera.perspective = view_dictionary['perspective']
    print(f"DEBUG: 14. Camera Perspective set to: {view_dictionary['perspective']}") # Tracepoint 14
    
    # --- Screenshot ---
    # NOTE: Corrected filename from 't{i}.png' to 't{frame_n}.png' as 'i' is the layer index.
    
    try:
        v.screenshot(path=screenshot_path)
        print(f"DEBUG: 15. Screenshot taken and saved to {screenshot_path}") # Tracepoint 15
    except Exception as e:
        print(f"ERROR: Screenshot failed at frame {frame_n}: {e}")
    


DEBUG: 1. Viewer initialized.


  0%|          | 0/75 [00:00<?, ?it/s]


DEBUG: 2. --- Starting Frame: 0 ---
DEBUG: 3. Frame 0: img_scale set to (1,1,1).
DEBUG: 6. Image layer added up to frame 0.
DEBUG: 7. Frame 0: Skipping track and camera update.

DEBUG: 2. --- Starting Frame: 1 ---

DEBUG: 2. --- Starting Frame: 2 ---

DEBUG: 2. --- Starting Frame: 3 ---

DEBUG: 2. --- Starting Frame: 4 ---

DEBUG: 2. --- Starting Frame: 5 ---

DEBUG: 2. --- Starting Frame: 6 ---

DEBUG: 2. --- Starting Frame: 7 ---

DEBUG: 2. --- Starting Frame: 8 ---

DEBUG: 2. --- Starting Frame: 9 ---

DEBUG: 2. --- Starting Frame: 10 ---

DEBUG: 2. --- Starting Frame: 11 ---

DEBUG: 2. --- Starting Frame: 12 ---

DEBUG: 2. --- Starting Frame: 13 ---

DEBUG: 2. --- Starting Frame: 14 ---

DEBUG: 2. --- Starting Frame: 15 ---

DEBUG: 2. --- Starting Frame: 16 ---

DEBUG: 2. --- Starting Frame: 17 ---

DEBUG: 2. --- Starting Frame: 18 ---

DEBUG: 2. --- Starting Frame: 19 ---

DEBUG: 2. --- Starting Frame: 20 ---

DEBUG: 2. --- Starting Frame: 21 ---

DEBUG: 2. --- Starting Frame: 22

/home/dayn/miniconda3/envs/macrohet_fixed/lib/python3.10/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (2, 6048, 6048) exceeds GL_MAX_TEXTURE_SIZE 2048 in at least one axis and will be downsampled. Rendering is currently in 3D mode.
  warnings.warn(


DEBUG: 10. NDisplay set to 3.
DEBUG: 11. Camera Center set to: (39.75, 3023.5, 3023.5)
DEBUG: 12. Camera Zoom set to: 0.139484126984127
DEBUG: 13. Camera Angles set to: (30.987817719483903, 35.572853672744216, 48.907463346476064)
DEBUG: 14. Camera Perspective set to: 0.0
DEBUG: 15. Screenshot taken and saved to /mnt/OPERA3/Nathan/hpig_tracking_animation/t31.png

DEBUG: 2. --- Starting Frame: 32 ---
DEBUG: 4. Frame > 0: img_scale set to (1,1,1). Starting layer pop.
DEBUG: 5. Layer at index 2 popped.
DEBUG: 5. Layer at index 1 popped.
DEBUG: 5. Layer at index 0 popped.


...

DEBUG: 6. Image layer added up to frame 72.
DEBUG: 8. Tracks filtered and converted to numpy. Shape: (16163, 4)
DEBUG: 9. Tracks layer added.
DEBUG: 10. NDisplay set to 3.
DEBUG: 11. Camera Center set to: (39.75, 3023.5, 3023.5)
DEBUG: 12. Camera Zoom set to: 0.139484126984127
DEBUG: 13. Camera Angles set to: (30.987817719483903, 35.572853672744216, 48.907463346476064)
DEBUG: 14. Camera Perspective set to: 0.0
DEBUG: 15. Screenshot taken and saved to /mnt/OPERA3/Nathan/hpig_tracking_animation/t72.png

DEBUG: 2. --- Starting Frame: 73 ---
DEBUG: 4. Frame > 0: img_scale set to (1,1,1). Starting layer pop.
DEBUG: 5. Layer at index 2 popped.
DEBUG: 5. Layer at index 1 popped.
DEBUG: 5. Layer at index 0 popped.


/home/dayn/miniconda3/envs/macrohet_fixed/lib/python3.10/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (2, 6048, 6048) exceeds GL_MAX_TEXTURE_SIZE 2048 in at least one axis and will be downsampled. Rendering is currently in 3D mode.
  warnings.warn(
/home/dayn/miniconda3/envs/macrohet_fixed/lib/python3.10/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (2, 6048, 6048) exceeds GL_MAX_TEXTURE_SIZE 2048 in at least one axis and will be downsampled. Rendering is currently in 3D mode.
  warnings.warn(


DEBUG: 6. Image layer added up to frame 73.
DEBUG: 8. Tracks filtered and converted to numpy. Shape: (16373, 4)
DEBUG: 9. Tracks layer added.
DEBUG: 10. NDisplay set to 3.
DEBUG: 11. Camera Center set to: (39.75, 3023.5, 3023.5)
DEBUG: 12. Camera Zoom set to: 0.139484126984127
DEBUG: 13. Camera Angles set to: (30.987817719483903, 35.572853672744216, 48.907463346476064)
DEBUG: 14. Camera Perspective set to: 0.0
DEBUG: 15. Screenshot taken and saved to /mnt/OPERA3/Nathan/hpig_tracking_animation/t73.png

DEBUG: 2. --- Starting Frame: 74 ---
DEBUG: 4. Frame > 0: img_scale set to (1,1,1). Starting layer pop.
DEBUG: 5. Layer at index 2 popped.
DEBUG: 5. Layer at index 1 popped.
DEBUG: 5. Layer at index 0 popped.


/home/dayn/miniconda3/envs/macrohet_fixed/lib/python3.10/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (2, 6048, 6048) exceeds GL_MAX_TEXTURE_SIZE 2048 in at least one axis and will be downsampled. Rendering is currently in 3D mode.
  warnings.warn(
/home/dayn/miniconda3/envs/macrohet_fixed/lib/python3.10/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (2, 6048, 6048) exceeds GL_MAX_TEXTURE_SIZE 2048 in at least one axis and will be downsampled. Rendering is currently in 3D mode.
  warnings.warn(


DEBUG: 6. Image layer added up to frame 74.
DEBUG: 8. Tracks filtered and converted to numpy. Shape: (16575, 4)
DEBUG: 9. Tracks layer added.
DEBUG: 10. NDisplay set to 3.
DEBUG: 11. Camera Center set to: (39.75, 3023.5, 3023.5)
DEBUG: 12. Camera Zoom set to: 0.139484126984127
DEBUG: 13. Camera Angles set to: (30.987817719483903, 35.572853672744216, 48.907463346476064)
DEBUG: 14. Camera Perspective set to: 0.0
DEBUG: 15. Screenshot taken and saved to /mnt/OPERA3/Nathan/hpig_tracking_animation/t74.png
